## Imports

In [1]:

%load_ext autoreload
%autoreload 2

import os
import sys
import torch
import einops
import random
import pickle
import numpy as np
import pprint as pp
import torch.nn as nn
from pathlib import Path
from pyfiglet import Figlet
from argparse import Namespace
import plotly.graph_objects as go
from torch.utils.data import DataLoader
from plotly.subplots import make_subplots
from src.plots import plot_losses, plot_field_comparison


# Add the project root directory to Python path
top_dir = Path.cwd().parent
sys.path.insert(0, str(top_dir.absolute()))

from src import *

slurm_dir = top_dir / 'slurms'
pickle_dir = top_dir / 'pickles'
data_dir = top_dir / 'datasets'


In [2]:

args = Namespace(**{'device': 'cpu', 'n_sensors': 10, 'data_rows': 180, 'data_cols': 360, 'input_length': 10, 'forecast_length': 1})
train_ds, val_ds, test_ds, _ = datasets.load_sst_data(args)

prediction, target = (train_ds[0][0][-1], train_ds[0][1][-1])

sensors = helpers.generate_sensor_positions(args.n_sensors*4, args.data_rows, args.data_cols)
with open(data_dir / 'sst' / 'SST_zeros.pkl', 'rb') as f:
    zeros = pickle.load(f)
sensors = [pos for pos in sensors if (zeros[pos[0], pos[1]] == False)]
sensors = sensors[0:args.n_sensors]


In [ ]:

plots.plot_field_comparison(prediction, target, 'sst', sensors=sensors, sensors_all=True, save=True, fname='test')


DONE


In [8]:

# Parameters
X = 6  # Number of dimensions (rows)
T = 100  # Number of timesteps (columns)
n_sin = 10 # Number of sinusoids

# Generate random data for demonstration (X rows × T columns)
data = helpers.generate_sinusoid_sum(n_sin, X, T, seed=0)
data = data + 10

# X-axis labels for each row
x_labels = ['Sensor 1 Value', 'Sensor 1 Position', 'Sensor 2 Value', 'Sensor 2 Position', 'Sensor N Value', 'Sensor N Position']

# Create subplots
fig = make_subplots(rows=X, cols=1, 
                   vertical_spacing=0.05)

# Add traces for each dimension
for i in range(X):
    fig.add_trace(
        go.Scatter(
            x=list(range(T)),  # Time steps on x-axis
            y=data[i, :],      # Data for this dimension
            mode='lines',
            name=f'Dim {i+1}',
            line=dict(color='red')
        ),
        row=i+1, col=1
    )

# Update layout
fig.update_layout(
    height=100*X,  # Adjust height based on number of subplots
    width=600,
    title_text=f"Time Series for {X} Dimensions",
    showlegend=False,
    plot_bgcolor='white',
    paper_bgcolor='white',
)

# Update axes for all subplots
for i in range(1, X+1):  # Subplot rows are 1-indexed
    # Set x-axis label
    fig.update_xaxes(
        title_text=x_labels[i-1],
        showticklabels=False, 
        ticks="", 
        row=i, 
        col=1,
        linecolor='black',
        linewidth=2
    )
    # Set y-axis properties
    fig.update_yaxes(
        showticklabels=False, 
        ticks="", 
        row=i, 
        col=1,
        linecolor='black',
        linewidth=2
    )

fig.write_image("figure1.pdf")
fig.show()


In [5]:

f = Figlet(font='ogre')
for dataset in ['planetswe_full', 'sst', 'plasma']:
    best_result = helpers.get_top_N_models_by_loss(dataset, pickle_dir, N=1)[0][1]
    plots.plot_model_results_scatter(results, dataset, 12, save=True, fname=f"{dataset}_scatter")


TypeError: string indices must be integers, not 'str'